# Week 2: Baseline modelling — Viscosity prediction

Predicting viscosity at different shear rates using baseline regression models.

**Targets:**
1. `Viscosity_at_shear_rate_1_1/s`
2. `Viscosity_at_shear_rate_10_1/s`
3. `Viscosity_at_shear_rate_100_1/s`
4. `Viscosity_avg` (mean of the three)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# --- Config ---
VISCOSITY_COLS = [
    "Viscosity_at_shear_rate_1_1/s",
    "Viscosity_at_shear_rate_10_1/s",
    "Viscosity_at_shear_rate_100_1/s",
]
TEST_SIZE = 0.25
RANDOM_STATE = 42

DATA_PATH = Path("../data/processed/combined_slurry_data_cleaned.csv")

In [3]:
df = pd.read_csv(DATA_PATH)

# Drop high-cardinality identifier
if "Composite_Mix_ID" in df.columns:
    df = df.drop(columns=["Composite_Mix_ID"])

# Create averaged viscosity target
df["Viscosity_avg"] = df[VISCOSITY_COLS].mean(axis=1)

# All viscosity targets (individual + averaged)
all_targets = VISCOSITY_COLS + ["Viscosity_avg"]

# Feature columns (everything except the viscosity targets)
feature_cols = [c for c in df.columns if c not in all_targets]
X_full = df[feature_cols]

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()

print(f"Rows: {len(df)}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"  Numeric: {num_cols}")
print(f"  Categorical: {cat_cols}")
print(f"\nTargets to predict: {all_targets}")

Rows: 178
Features (5): ['Formulation', 'Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'Source_Batch']
  Numeric: ['Solid_Content_pct', 'Solid_Additive_pct']
  Categorical: ['Formulation', 'Dispersent_Type', 'Source_Batch']

Targets to predict: ['Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s', 'Viscosity_avg']


In [4]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso": Lasso(alpha=0.1, max_iter=5000, random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "DecisionTree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

all_results = []

for target in all_targets:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    for name, model in models.items():
        pipe = Pipeline(
            steps=[
                ("preprocess", preprocess),
                ("model", model),
            ]
        )
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        
        all_results.append({
            "target": target.replace("Viscosity_at_shear_rate_", "SR_").replace("1/s", ""),
            "model": name,
            "R²": r2,
            "RMSE": rmse,
            "MAE": mae,
        })

results_df = pd.DataFrame(all_results)
results_df

,target,model,R²,RMSE,MAE
0,SR_1_,Dummy (mean),-0.244719,52.385980,48.020681
1,SR_1_,Ridge,0.365674,37.396899,28.190253
2,SR_1_,Lasso,0.362060,37.503271,28.311527
3,SR_1_,KNN (k=5),0.289995,39.564910,27.185836
4,SR_1_,DecisionTree,0.560762,31.119264,17.965290
5,SR_1_,RandomForest,0.603805,29.555214,17.109287
6,SR_10_,Dummy (mean),-0.185490,10.345020,8.871048
7,SR_10_,Ridge,0.690438,5.286357,4.209084
8,SR_10_,Lasso,0.696793,5.231811,4.249626
9,SR_10_,KNN (k=5),0.673404,5.429848,4.116977


In [5]:
print("=" * 60)
print("BASELINE REGRESSION RESULTS (sorted by R² per target)")
print("=" * 60)

for target in results_df["target"].unique():
    subset = results_df[results_df["target"] == target].sort_values("R²", ascending=False)
    print(f"\n▶ {target}")
    display(subset.style.format({"R²": "{:.3f}", "RMSE": "{:.2f}", "MAE": "{:.2f}"}).hide(axis="index"))

# Summary: best model per target
print("\n" + "=" * 60)
print("BEST MODEL PER TARGET (by R²)")
print("=" * 60)
best_per_target = results_df.loc[results_df.groupby("target")["R²"].idxmax()]
display(best_per_target.style.format({"R²": "{:.3f}", "RMSE": "{:.2f}", "MAE": "{:.2f}"}).hide(axis="index"))

BASELINE REGRESSION RESULTS (sorted by R² per target)

▶ SR_1_


target,model,R²,RMSE,MAE
SR_1_,RandomForest,0.604,29.56,17.11
SR_1_,DecisionTree,0.561,31.12,17.97
SR_1_,Ridge,0.366,37.40,28.19
SR_1_,Lasso,0.362,37.50,28.31
SR_1_,KNN (k=5),0.290,39.56,27.19
SR_1_,Dummy (mean),-0.245,52.39,48.02



▶ SR_10_


target,model,R²,RMSE,MAE
SR_10_,RandomForest,0.881,3.28,2.15
SR_10_,DecisionTree,0.868,3.46,2.24
SR_10_,Lasso,0.697,5.23,4.25
SR_10_,Ridge,0.690,5.29,4.21
SR_10_,KNN (k=5),0.673,5.43,4.12
SR_10_,Dummy (mean),-0.185,10.35,8.87



▶ SR_100_


target,model,R²,RMSE,MAE
SR_100_,RandomForest,0.903,0.89,0.57
SR_100_,DecisionTree,0.892,0.94,0.59
SR_100_,Ridge,0.851,1.10,0.90
SR_100_,Lasso,0.825,1.19,0.98
SR_100_,KNN (k=5),0.763,1.39,1.05
SR_100_,Dummy (mean),-0.145,3.05,2.62



▶ Viscosity_avg


target,model,R²,RMSE,MAE
Viscosity_avg,RandomForest,0.693,10.84,6.41
Viscosity_avg,DecisionTree,0.663,11.35,6.85
Viscosity_avg,Lasso,0.473,14.20,10.74
Viscosity_avg,Ridge,0.473,14.21,10.68
Viscosity_avg,KNN (k=5),0.407,15.07,10.71
Viscosity_avg,Dummy (mean),-0.234,21.74,19.68



BEST MODEL PER TARGET (by R²)


target,model,R²,RMSE,MAE
SR_100_,RandomForest,0.903,0.89,0.57
SR_10_,RandomForest,0.881,3.28,2.15
SR_1_,RandomForest,0.604,29.56,17.11
Viscosity_avg,RandomForest,0.693,10.84,6.41
